In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

Имеется информация за $10$ дней в агрегаторе такси:

- Количество заказов = $8770$.
- Среднее время поиска и назначения водителя на заказ составляет $2$ минуты и $1$ секунду (стандартное отклонение — $32$ секунды).
- Конверсия из заказа в назначение водителя на заказ равна $94\%$.
- Среднее время подачи такси после назначения водителя на заказ составляет $5$ минут и $16$ секунд (стандартное отклонение — $1$ минута и $5$ секунд).
- Конверсия из назначения в прибытие автомобиля в точку А составляет $82\%$.
- Среднее время поездки (от точки А до точки В) составляет $24$ минуты и $36$ секунд (стандартное отклонение — $7$ минут и $2$ секунды).
- Конверсия из прибытия автомобиля в точку А в прибытие в точку В составляет $88\%$.

### Задача 1. Автоматизация доверительного интервала (3 балла)

Создай функцию, которая будет проводить автоматизированный расчёт границ доверительного интервала для метрики.

Функция берёт на вход:
- Среднее значение метрики
- Стандартное отклонение метрики
- Количество наблюдений
- Уровень значимости $\alpha$ со значением по умолчанию $5\%$

Функция выдаёт:
- Левую границу доверительного интервала
- Правую границу доверительного интервала

Примени функцию к набору метрик:

- Конверсия из заказа в назначение водителя на заказ.
- Конверсия из назначения в прибытие водителя в точку А.
- Конверсия из прибытия в точку А в прибытие в точку В.
- Время между созданием заказа и назначением водителя на заказ.
- Время между назначением водителя на заказ и прибытием в точку А.
- Время между прибытием в точку А и прибытием в точку В.

*Квантиль нормального стандартного распределения необходимо рассчитать внутри функции.*

In [13]:
def confidence_interval(mean, std, n, alpha=0.05):
    """Двусторонний нормальный ДИ среднего по сводным данным."""
    if n <= 0 or std < 0 or not 0 < alpha < 1:
        raise ValueError("Нужны n > 0, std >= 0 и 0 < alpha < 1")
    z = stats.norm.ppf(1 - alpha / 2)
    error = z * std / np.sqrt(n)
    return mean - error, mean + error

n_orders = 8770
p1, p2, p3 = 0.94, 0.82, 0.88

n_assigned = n_orders * p1
n_arrived = n_assigned * p2
n_completed = n_arrived * p3

conversions = np.array([p1, p2, p3])
n_conversion = np.array([n_orders, n_assigned, n_arrived])
means_sec = np.array([2 * 60 + 1, 5 * 60 + 16, 24 * 60 + 36])
stds_sec = np.array([32, 60 + 5, 7 * 60 + 2])
n_time = np.array([n_assigned, n_arrived, n_completed])
stages = ['Заказ -> назначение', 'Назначение -> точка А', 'Точка А -> точка В']

rows = []
for stage, p_hat, n in zip(stages, conversions, n_conversion):
    low, high = confidence_interval(p_hat, np.sqrt(p_hat * (1 - p_hat)), n)
    rows.append({'Метрика': stage, 'n': n, 'Оценка, %': 100 * p_hat,
                 'ДИ: левая, %': 100 * low, 'ДИ: правая, %': 100 * high})
conversion_results = pd.DataFrame(rows)
display(conversion_results.round(4))

rows = []
for stage, mean, std, n in zip(stages, means_sec, stds_sec, n_time):
    low, high = confidence_interval(mean, std, n)
    rows.append({'Метрика': stage, 'n': n, 'Среднее, с': mean,
                 'ДИ: левая, с': low, 'ДИ: правая, с': high})
time_results = pd.DataFrame(rows)
display(time_results.round(4))

,Метрика,n,"Оценка, %","ДИ: левая, %","ДИ: правая, %"
0,Заказ -> назначение,8770.000,94.0,93.5030,94.4970
1,Назначение -> точка А,8243.800,82.0,81.1707,82.8293
2,Точка А -> точка В,6759.916,88.0,87.2253,88.7747


,Метрика,n,"Среднее, с","ДИ: левая, с","ДИ: правая, с"
0,Заказ -> назначение,8243.8000,121,120.3092,121.6908
1,Назначение -> точка А,6759.9160,316,314.4505,317.5495
2,Точка А -> точка В,5948.7261,1476,1465.2762,1486.7238


## Задача 2. Среднее время до точки В (2 балла)

Можно ли утверждать, что среднее время от создания заказа до прибытия в точку **В** составляет не более $35$ минут?

<br>

**Подсказка**

Для ответа на вопрос рассчитайте границы доверительного интервала для суммы трёх показателей времени.

Для этого:
- Рассчитайте среднее суммы трёх показателей времени.
- Рассчитайте стандартную ошибку суммы трёх средних.
- Определите количество наблюдений для каждого показателя.
- Постройте доверительный интервал.

В качестве уровня значимости возьмите $5\%$.

In [14]:
alpha = 0.05
mean_total = means_sec.sum()
se_total = np.sqrt(np.sum(stds_sec**2 / n_time))
z = stats.norm.ppf(1 - alpha / 2)
ci_total = (mean_total - z * se_total, mean_total + z * se_total)

print(f'Среднее: {mean_total / 60:.4f}')
print(f'Стандартная ошибка суммы средних: {se_total:.4f} с')
print(f'95% ДИ: [{ci_total[0] / 60:.4f}; {ci_total[1] / 60:.4f}]')
print('Верхняя граница ниже 35 минут:', ci_total[1] < 35 * 60)

Среднее: 31.8833
Стандартная ошибка суммы средних: 5.5395 с
95% ДИ: [31.7024; 32.0643]
Верхняя граница ниже 35 минут: True


**Ответ.** Доверительный интервал среднего лежит ниже 35 минут, поэтому данные подтверждают утверждение о среднем времени.

## Задача 3. Ожидаемая выручка (2 балла)

Предположите, что все заказы были равномерно распределены по $10$ дням, а конверсия была неизменной в течение всего периода времени.

Средняя стоимость одного заказа составляет $750$ рублей.

Какую максимальную ожидаемую выручку за один среднестатистический день можно получить, если использовать правую границу $95\%$ доверительного интервала для конверсии из заказа в завершённую поездку?

В качестве уровня значимости возьмите $5\%$.

Округлите ответ до $0$ знаков после запятой.

<br>

*Подсказка*

Рассчитайте границы доверительного интервала для конверсии из заказа в завершённую поездку, которая является произведением трёх звеньев конверсии.

In [15]:
p_total = p1 * p2 * p3
ci_conversion_total = confidence_interval(
    p_total, np.sqrt(p_total * (1 - p_total)), n_orders
)
orders_per_day = n_orders / 10
price = 750
revenue_point = orders_per_day * price * p_total
revenue_upper = orders_per_day * price * ci_conversion_total[1]

print(f'Конверсия в завершённую поездку: {p_total:.4%}')
print(f'95% ДИ конверсии: [{ci_conversion_total[0]:.4%}; {ci_conversion_total[1]:.4%}]')
print(f'Заказов в день: {orders_per_day:.0f}')
print(f'Точечная оценка дневной выручки: {revenue_point:.0f} руб.')
print(f'Ответ по верхней границе ДИ: {revenue_upper:.0f} руб.')

Конверсия в завершённую поездку: 67.8304%
95% ДИ конверсии: [66.8528%; 68.8080%]
Заказов в день: 877
Точечная оценка дневной выручки: 446154 руб.
Ответ по верхней границе ДИ: 452585 руб.


## Задача 4. Бутстрап для времени поездки (3 балла)

Предположим, что время поездки подчиняется логнормальному распределению.

Это типично для показателей, которые не могут быть отрицательными и имеют длинный правый "хвост" — небольшое количество очень долгих поездок.

Параметры распределения:

- $\mu = 3.0$
- $\sigma = 0.8$

**Задание**

1. Сгенерируйте исходную выборку размера $n = 1000$ из логнормального распределения с заданными параметрами. Это будет наша "реальная" выборка времени поездок в минутах.

2. Реализуйте бутстрап-процедуру:

   - Сгенерируйте $B = 5000$ бутстрап-выборок — повторных выборок с возвращением — того же размера $n$ из исходной выборки.
   - Для каждой бутстрап-выборки вычислите целевые статистики:
     - Среднее;
     - Медиану;
     - Дисперсию.

In [ ]:
rng = np.random.default_rng(42)
mu, sigma = 3.0, 0.8
n, B = 1000, 5000
trip_times = rng.lognormal(mean=mu, sigma=sigma, size=n)

boot_means = np.empty(B)
boot_medians = np.empty(B)
boot_variances = np.empty(B)

for i in range(B):
    sample = rng.choice(trip_times, size=n, replace=True)
    boot_means[i] = sample.mean()
    boot_medians[i] = np.median(sample)
    boot_variances[i] = sample.var(ddof=1)

bootstrap_results = pd.DataFrame({
    'Среднее': boot_means,
    'Медиана': boot_medians,
    'Дисперсия': boot_variances,
})
print(f'Размер исходной выборки: {trip_times.size}')
print(f'Количество бутстрап-выборок: {len(bootstrap_results)}')
print("\nРезультат бутстрап: (первые 5 строк)")
display(bootstrap_results.head())

Размер исходной выборки: 1000
Количество бутстрап-выборок: 5000

Результат бутстрап: (первые 5 строк)


,Среднее,Медиана,Дисперсия
0,26.805077,19.935426,550.336704
1,27.532321,19.298633,745.382467
2,28.118117,21.221183,582.072505
3,26.884174,19.892291,677.027948
4,26.311374,19.640898,594.225176
